# 🏆 Notebook 7 — Model Comparison
**Goal:** Compare all models side by side using MAE, RMSE, and MAPE. Produce a final comparison table and radar chart.

> **Prerequisites:** Run notebooks 3 → 6 first so metric JSON files exist in `models/`.

**Contents:**
1. Load All Metrics
2. Metrics Table
3. Bar Chart Comparison
4. Radar Chart
5. Best Model Recommendation


## 1. Load All Metrics

In [ ]:
import os, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs("plots", exist_ok=True)
BLUE,RED,ORANGE,GREEN,PURPLE,BROWN = "#1A6FBF","#D62728","#E07B39","#2CA02C","#9467BD","#8C564B"
COLORS = [BLUE,ORANGE,GREEN,RED,PURPLE,BROWN,"#E377C2","#17BECF","#BCBD22","#7F7F7F"]
plt.rcParams.update({"figure.facecolor":"#FAFAFA","axes.facecolor":"#FAFAFA",
    "axes.grid":True,"grid.alpha":0.3,"font.family":"DejaVu Sans",
    "axes.spines.top":False,"axes.spines.right":False})

records = []
for path in glob.glob("models/*_metrics.json"):
    with open(path) as f:
        records.append(json.load(f))

df_metrics = pd.DataFrame(records)

# Tag model type for easier grouping
def tag_type(name):
    n = str(name).lower()
    if "arima" in n and "sarimax" not in n and "sarima" not in n: return "Statistical"
    if "sarima" in n or "sarimax" in n: return "Statistical"
    if "prophet" in n: return "Statistical/ML"
    if "lstm" in n: return "Deep Learning"
    return "Baseline"

df_metrics["type"] = df_metrics["model"].apply(tag_type)

print(f"Loaded {len(df_metrics)} model runs across types: {df_metrics['type'].value_counts().to_dict()}")
df_metrics[["model","type","target","MAE","RMSE","MAPE"]].sort_values(["target","MAPE"])


## 2. Metrics Tables

In [ ]:
for keyword in ["Volume","Value"]:
    sub = df_metrics[df_metrics["target"].str.contains(keyword,case=False)].sort_values("MAPE").reset_index(drop=True)
    sub.index += 1
    print(f"\n{'='*60}")
    print(f"  {keyword} — Ranked by MAPE")
    print(f"{'='*60}")
    print(sub[["model","MAE","RMSE","MAPE"]].to_string())
    best = sub.iloc[0]
    print(f"\n  ★ Best: {best['model']}  MAPE={best['MAPE']:.2f}%")


## 3. Bar Chart Comparison

In [ ]:
COLORS = [BLUE,ORANGE,GREEN,RED,PURPLE,BROWN]

for keyword, tag in [("Volume","vol"),("Value","val")]:
    sub = df_metrics[df_metrics["target"].str.contains(keyword,case=False)].sort_values("MAPE").reset_index(drop=True)
    models = sub["model"].tolist()
    colors_ = COLORS[:len(models)]

    fig, axes = plt.subplots(1,3,figsize=(18,5))
    for ax, (col, label) in zip(axes,[("MAE","MAE"),("RMSE","RMSE"),("MAPE","MAPE (%)")]):
        vals = sub[col].tolist()
        bars = ax.bar(range(len(models)), vals, color=colors_, edgecolor="white", width=0.6)
        ax.set_xticks(range(len(models)))
        ax.set_xticklabels(models, rotation=20, ha="right", fontsize=9)
        ax.set_ylabel(label); ax.set_title(label,fontweight="bold")
        for bar,val in zip(bars,vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vals)*0.01,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=9)
        bars[vals.index(min(vals))].set_edgecolor("gold")
        bars[vals.index(min(vals))].set_linewidth(2.5)
    fig.suptitle(f"{keyword} — Model Comparison (gold border = best)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"plots/19_{tag}_comparison.png",dpi=150,bbox_inches="tight")
    plt.show()


## 4. Radar Chart — Volume

In [ ]:
sub = df_metrics[df_metrics["target"].str.contains("Volume",case=False)].sort_values("MAPE").reset_index(drop=True)

for col in ["MAE","RMSE","MAPE"]:
    mn,mx = sub[col].min(), sub[col].max()
    sub[f"{col}_norm"] = 1 - ((sub[col]-mn)/(mx-mn+1e-9))

categories = ["MAE","RMSE","MAPE"]
N = len(categories)
angles = [n/float(N)*2*np.pi for n in range(N)] + [0]

fig, ax = plt.subplots(figsize=(8,8), subplot_kw={"polar":True})
for i,(_, row) in enumerate(sub.iterrows()):
    values = [row["MAE_norm"],row["RMSE_norm"],row["MAPE_norm"],row["MAE_norm"]]
    ax.plot(angles, values, lw=2, color=COLORS[i%len(COLORS)], label=row["model"])
    ax.fill(angles, values, alpha=0.07, color=COLORS[i%len(COLORS)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_title("Radar Chart — Volume Models\n(Higher = Better, Normalised)",
             fontsize=12, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35,1.1), fontsize=9)
plt.tight_layout()
plt.savefig("plots/19_radar_comparison.png",dpi=150,bbox_inches="tight")
plt.show()


## 5. Best Model Recommendation

In [ ]:
print("\n" + "="*60)
print("  FINAL RECOMMENDATION")
print("="*60)
for keyword in ["Volume","Value"]:
    sub = df_metrics[df_metrics["target"].str.contains(keyword,case=False)].sort_values("MAPE")
    best = sub.iloc[0]
    print(f"\n  {keyword}:")
    print(f"    Best model : {best['model']}")
    print(f"    MAE        : {best['MAE']:.4f}")
    print(f"    RMSE       : {best['RMSE']:.4f}")
    print(f"    MAPE       : {best['MAPE']:.2f}%")

df_metrics[["model","target","MAE","RMSE","MAPE"]].to_csv("models/all_metrics_comparison.csv",index=False)
print("\n✅ Full comparison saved to models/all_metrics_comparison.csv")


## 5. Full Comparison — Baselines vs Models

In [ ]:
for keyword in ["Volume","Value"]:
    sub = df_metrics[df_metrics["target"].str.contains(keyword,case=False)].sort_values("MAPE").reset_index(drop=True)
    sub.index += 1

    # colour by type
    type_colors = {"Baseline": "lightgray", "Statistical": BLUE,
                   "Statistical/ML": ORANGE, "Deep Learning": GREEN}
    bar_colors = [type_colors.get(t, RED) for t in sub["type"]]

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    for ax, col in zip(axes, ["MAE","RMSE","MAPE"]):
        vals = sub[col].tolist()
        bars = ax.bar(range(len(sub)), vals, color=bar_colors, edgecolor="white", width=0.65)
        ax.set_xticks(range(len(sub)))
        ax.set_xticklabels(sub["model"], rotation=30, ha="right", fontsize=8)
        ax.set_ylabel(col); ax.set_title(col, fontweight="bold")
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vals)*0.008,
                    f"{val:.1f}", ha="center", va="bottom", fontsize=7.5)
        # Gold border on best
        best_i = vals.index(min(vals))
        bars[best_i].set_edgecolor("gold"); bars[best_i].set_linewidth(2.5)

    # Legend for type colours
    from matplotlib.patches import Patch
    legend_els = [Patch(fc=c, label=t) for t, c in type_colors.items()]
    axes[0].legend(handles=legend_els, fontsize=8, title="Model Type")

    fig.suptitle(f"{keyword} — All Models vs Baselines (gold = best  |  gray = baselines)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"plots/19_full_comparison_{keyword.lower()}.png", dpi=150, bbox_inches="tight")
    plt.show()


## 6. Improvement Over Best Baseline

In [ ]:
for keyword in ["Volume","Value"]:
    sub = df_metrics[df_metrics["target"].str.contains(keyword,case=False)].copy()

    baseline_best_mape = sub[sub["type"]=="Baseline"]["MAPE"].min()
    sub["Improvement over best baseline (MAPE%)"] = (
        (baseline_best_mape - sub["MAPE"]) / baseline_best_mape * 100
    ).round(2)

    # Separate tables
    baselines = sub[sub["type"]=="Baseline"].sort_values("MAPE").reset_index(drop=True)
    models    = sub[sub["type"]!="Baseline"].sort_values("MAPE").reset_index(drop=True)

    print(f"\n{'='*70}\n  {keyword} — Baselines\n{'='*70}")
    print(baselines[["model","MAE","RMSE","MAPE"]].to_string(index=False))

    print(f"\n{'='*70}\n  {keyword} — Forecasting Models (ranked by MAPE)\n{'='*70}")
    print(models[["model","MAE","RMSE","MAPE","Improvement over best baseline (MAPE%)"]].to_string(index=False))

    best_model    = models.iloc[0]
    best_baseline = baselines.iloc[0]
    improvement   = best_model["Improvement over best baseline (MAPE%)"]
    print(f"\n  Best model    : {best_model['model']}  MAPE={best_model['MAPE']:.2f}%")
    print(f"  Best baseline : {best_baseline['model']}  MAPE={best_baseline['MAPE']:.2f}%")
    if improvement > 0:
        print(f"  ✅ {improvement:.1f}% improvement in MAPE over best baseline")
    else:
        print(f"  ⚠️  Models FAILED to beat the best baseline — revisit tuning!")


## 7. Radar Chart — All Models

In [ ]:
for keyword in ["Volume","Value"]:
    sub = df_metrics[df_metrics["target"].str.contains(keyword,case=False)].sort_values("MAPE").reset_index(drop=True)

    for col in ["MAE","RMSE","MAPE"]:
        mn, mx = sub[col].min(), sub[col].max()
        sub[f"{col}_norm"] = 1 - ((sub[col]-mn)/(mx-mn+1e-9))

    categories = ["MAE","RMSE","MAPE"]
    angles = [n/3*2*3.14159 for n in range(3)] + [0]

    fig, ax = plt.subplots(figsize=(8,8), subplot_kw={"polar":True})
    type_colors_radar = {"Baseline":"gray","Statistical":BLUE,"Statistical/ML":ORANGE,"Deep Learning":GREEN}

    for i, (_, row) in enumerate(sub.iterrows()):
        vals = [row["MAE_norm"],row["RMSE_norm"],row["MAPE_norm"],row["MAE_norm"]]
        color = type_colors_radar.get(row["type"], RED)
        ls    = "--" if row["type"]=="Baseline" else "-"
        ax.plot(angles, vals, lw=1.8 if row["type"]!="Baseline" else 1.0,
                color=color, linestyle=ls, label=row["model"])
        ax.fill(angles, vals, alpha=0.04 if row["type"]=="Baseline" else 0.07, color=color)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=11)
    ax.set_title(f"Radar — {keyword}\n(Higher = Better, Normalised | dashed = baselines)",
                 fontsize=11, fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.45,1.15), fontsize=8)
    plt.tight_layout()
    plt.savefig(f"plots/19_radar_{keyword.lower()}.png", dpi=150, bbox_inches="tight")
    plt.show()


## 8. Final Summary & Save

In [ ]:
print("\n" + "="*65)
print("  FINAL SUMMARY")
print("="*65)

for keyword in ["Volume","Value"]:
    sub = df_metrics[df_metrics["target"].str.contains(keyword,case=False)].sort_values("MAPE")
    best_model    = sub[sub["type"]!="Baseline"].iloc[0]
    best_baseline = sub[sub["type"]=="Baseline"].iloc[0]
    imp = ((best_baseline["MAPE"] - best_model["MAPE"]) / best_baseline["MAPE"] * 100)

    print(f"\n  {keyword}:")
    print(f"    Best model    : {best_model['model']:<45} MAPE = {best_model['MAPE']:.2f}%")
    print(f"    Best baseline : {best_baseline['model']:<45} MAPE = {best_baseline['MAPE']:.2f}%")
    print(f"    Improvement   : {imp:.1f}%")

df_metrics[["model","type","target","MAE","RMSE","MAPE"]].sort_values(
    ["target","MAPE"]).to_csv("models/all_metrics_comparison.csv",index=False)
print("\n✅ Full comparison saved to models/all_metrics_comparison.csv")
